# Build and Query a Feature Store for ML Training & Serving

## Data prepearing

In [1]:
!pip install feast pandas pyarrow scikit-learn

INFO: pip is looking at multiple versions of uvicorn-worker to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.1/513.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.46.0
    Uninstalling uvicorn-0.46.0:
      Successfully uninst

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'paysim1' dataset.
Path to dataset files: /kaggle/input/paysim1


In [3]:
import pandas as pd
import os
files = os.listdir(path)
#print(files)

df = pd.read_csv(os.path.join(path,'PS_20174392719_1491204439457_log.csv'))
df.head(3)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


In [4]:

df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [5]:
df.dtypes

,0
step,int64
type,object
amount,float64
nameOrig,object
oldbalanceOrg,float64
newbalanceOrig,float64
nameDest,object
oldbalanceDest,float64
newbalanceDest,float64
isFraud,int64


In [6]:
df.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [7]:
df['step'].value_counts()

,count
step,
19,51352
18,49579
187,49083
235,47491
307,46968
...,...
725,4
655,4
246,4


In [8]:
df['isFraud'].value_counts()

,count
isFraud,
0,6354407
1,8213


In [16]:
df.shape

(6362620, 11)

In [9]:
df['isFlaggedFraud'].value_counts()

,count
isFlaggedFraud,
0,6362604
1,16


In [10]:
df['type'].value_counts()

,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [11]:
df['nameOrig'].value_counts()

,count
nameOrig,
C1530544995,3
C545315117,3
C724452879,3
C1784010646,3
C1677795071,3
...,...
C1567523029,1
C644777639,1
C1256645416,1


In [12]:
fraud_customers = (
    df[df["isFraud"] == 1]["type"].unique())
print(f"Say {len(fraud_customers)}")
print(f"Tipler{fraud_customers}")

Say 2
Tipler['TRANSFER' 'CASH_OUT']


In [13]:
df.head(2)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0


In [18]:
fraud     = df[df["isFraud"] == 1]          #hamısını götürek
non_fraud = df[df["isFraud"] == 0].sample(n=50_000, random_state=42)

df= pd.concat([fraud, non_fraud]).reset_index(drop=True)

# Feature Engineering

In [20]:
#Müştəri səviyyəsində aggregation
agg = df.groupby("nameOrig").agg(
    txn_count = ("type","count"),

    total_amount = ("amount","sum"),
    avg_amount = ("amount","mean"),
    max_amount = ("amount","max"),


    cashout_count = ("type", lambda x: (x=="CASH_OUT").sum()),
    transfer_count = ("type", lambda x: (x=="TRANSFER").sum()),
    payment_count  = ("type", lambda x: (x=="PAYMENT").sum()),

    balance_drain_count= ("newbalanceOrig",lambda x: (x==0).sum()),
    avg_balance_before = ("oldbalanceOrg", "mean"),

    dest_unchanged_count=("newbalanceDest",lambda x: (x == df.loc[x.index,"oldbalanceDest"]).sum()),

    past_fraud_count   = ("isFraud",       "sum"),
    was_flagged        = ("isFlaggedFraud", "max"),
    ).reset_index()

In [22]:
agg.head(1)

,nameOrig,txn_count,total_amount,avg_amount,max_amount,cashout_count,transfer_count,payment_count,balance_drain_count,avg_balance_before,dest_unchanged_count,past_fraud_count,was_flagged
0,C1000013879,1,516459.01,516459.01,516459.01,1,0,0,1,153435.6,0,0,0


In [23]:
#Nisbet featurelar

agg["cashout_ratio"]      = (agg["cashout_count"]/ agg["txn_count"]).round(4)
agg["transfer_ratio"]     = (agg["transfer_count"]/ agg["txn_count"]).round(4)
agg["balance_drain_ratio"]= (agg["balance_drain_count"]/ agg["txn_count"]).round(4)
agg["fraud_rate"]         = (agg["past_fraud_count"]/ agg["txn_count"]).round(4)

agg["dest_unchanged_ratio"]=(agg["dest_unchanged_count"]/agg["txn_count"]).round(4)
agg["avg_amount"]         = agg["avg_amount"].round(2)

In [24]:
agg = agg.rename(columns={"nameOrig": "customer_id"})

In [25]:
agg.head(2)

,customer_id,txn_count,total_amount,avg_amount,max_amount,cashout_count,transfer_count,payment_count,balance_drain_count,avg_balance_before,dest_unchanged_count,past_fraud_count,was_flagged,cashout_ratio,transfer_ratio,balance_drain_ratio,fraud_rate,dest_unchanged_ratio
0,C1000013879,1,516459.01,516459.01,516459.01,1,0,0,1,153435.60,0,0,0,1.0,0.0,1.0,0.0,0.0
1,C1000036340,1,253648.68,253648.68,253648.68,0,1,0,1,253648.68,1,1,0,0.0,1.0,1.0,1.0,1.0


In [26]:
agg.isnull().sum()

,0
customer_id,0
txn_count,0
total_amount,0
avg_amount,0
max_amount,0
cashout_count,0
transfer_count,0
payment_count,0
balance_drain_count,0
avg_balance_before,0


In [27]:
agg.shape

(58213, 18)

In [28]:
#Fraud olan
print(agg[agg["fraud_rate"]>0][
    ["customer_id","txn_count","avg_amount","balance_drain_ratio","fraud_rate"]
].head(5))

    customer_id  txn_count  avg_amount  balance_drain_ratio  fraud_rate
1   C1000036340          1   253648.68                  1.0         1.0
5   C1000086512          1    33676.59                  1.0         1.0
15  C1000331499          1  2016790.84                  1.0         1.0
22  C1000484178          1  3018810.85                  1.0         1.0
24  C1000513158          1    40388.58                  1.0         1.0


## Transaction-level features
transaction entity üçün
Feature store-da birdən çox entity ola bilər
Burada transaction-level feature-lar da saxlayırıq

In [29]:
aggt= df[["nameOrig","step","type","amount",
                                 "oldbalanceOrg","newbalanceOrig",
                                 "oldbalanceDest","newbalanceDest","isFraud"]].copy()

In [30]:
aggt["balance_delta_orig"] = (aggt["newbalanceOrig"] - aggt["oldbalanceOrg"]).round(2)
aggt["balance_delta_dest"] = (aggt["newbalanceDest"] - aggt["oldbalanceDest"]).round(2)

In [31]:
aggt["amount_to_balance_ratio"] = (aggt["amount"] /(aggt["oldbalanceOrg"] + 1e-3)).round(4)

aggt["is_cashout"]   = (aggt["type"]=="CASH_OUT").astype(int)
aggt["is_transfer"]  = (aggt["type"]=="TRANSFER").astype(int)

aggt["dest_unchanged"] = (aggt["newbalanceDest"] == aggt["oldbalanceDest"]).astype(int)

In [32]:
aggt.head(2)

,nameOrig,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_delta_orig,balance_delta_dest,amount_to_balance_ratio,is_cashout,is_transfer,dest_unchanged
0,C1305486145,1,TRANSFER,181.0,181.0,0.0,0.0,0.0,1,-181.0,0.0,1.0,0,1,1
1,C840083671,1,CASH_OUT,181.0,181.0,0.0,21182.0,0.0,1,-181.0,-21182.0,1.0,1,0,0


# Parquet - Feast Offline Store

In [33]:
os.makedirs("data", exist_ok=True)

#Feast üçün timestamp sütunları
# step sütununu real tarixə çevir step 1 = 2024-01-01 saat 01:00
base_date = pd.Timestamp("2024-01-01", tz="UTC")

In [34]:
agg["event_timestamp"] = base_date +pd.to_timedelta(
    df.groupby("nameOrig")["step"].max().reindex(
        agg["customer_id"]
    ).values, unit="h")
#son əməliyyatının step-i üzrə timestamp
agg["created"] = pd.Timestamp.now(tz="UTC")

In [35]:
aggt["event_timestamp"] = base_date + pd.to_timedelta(
    aggt["step"], unit="h")

aggt["created"] = pd.Timestamp.now(tz="UTC")


### Tip duzeltme

In [38]:
float_cols = ["avg_amount","max_amount","total_amount",
              "avg_balance_before","cashout_ratio","dest_unchanged_ratio",
              "transfer_ratio",
              "balance_drain_ratio","fraud_rate"]

for col in float_cols:
    agg[col] = agg[col].astype(float)

In [39]:
int_cols = ["txn_count","cashout_count","transfer_count","payment_count",
            "balance_drain_count","dest_unchanged_count",
            "past_fraud_count","was_flagged"]
for col in int_cols:
    agg[col] = agg[col].astype("int64")

In [40]:
#Parquete yazaq
agg.to_parquet("data/agg.parquet", index=False)

aggt.to_parquet("data/aggt.parquet", index=False)

In [41]:
#Yoxlayaq
for fname in ["agg.parquet", "aggt.parquet"]:
 size_kb = os.path.getsize(f"data/{fname}") / 1024
 df_test = pd.read_parquet(f"data/{fname}")

 print(f" {df_test.shape},{size_kb:.0f} KB")

 (58213, 20),2300 KB
 (58213, 17),3315 KB


In [42]:
 df_test['event_timestamp'].dtype

datetime64[ns, UTC]

##Feast Repo

Feast üçün iki fayl lazımdır:

Fayl (feature_store.yaml) - Backend konfiq (offline/online store)

Məqsəd (features.py) -  Entity, FeatureView, FeatureService definitionları

In [108]:
os.makedirs("feature_repo/data", exist_ok=True)

# feature_store.yaml
# Online store SQLite (development) — productionda Redis
# Offline store file (Parquet)

In [109]:
import os

repo_path = os.path.abspath("feature_repo")
db_path   = os.path.join(repo_path, "data", "registry.db")

yaml_config = f'''project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:///{db_path}
provider: local
online_store:
  type: sqlite
  path: {os.path.join(repo_path, "data", "online_store.db")}
offline_store:
  type: file
entity_key_serialization_version: 3
'''



In [110]:
with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(yaml_config)

print(yaml_config)

project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:////content/feature_repo/data/registry.db
provider: local
online_store:
  type: sqlite
  path: /content/feature_repo/data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3



## Feature Views

In [111]:
from pathlib import Path
actual_data_dir = os.path.abspath("data")
print("Parquet faylları", actual_data_dir)

Parquet faylları /content/data


In [112]:


features_py = f'''from datetime import timedelta
from feast import Entity, FeatureView, FeatureService, Field
from feast.types import Float64, Int64
from feast.infra.offline_stores.file_source import FileSource

data_dir = "{actual_data_dir}"

customer = Entity(
    name="customer",
    join_keys=["customer_id"],)

customer_source = FileSource(
    name="customer_features_source",
    path=data_dir + "/agg.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",)

customer_behavior_fv = FeatureView(
    name="customer_behavior",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="txn_count",      dtype=Int64),
        Field(name="total_amount",   dtype=Float64),
        Field(name="avg_amount",     dtype=Float64),
        Field(name="max_amount",     dtype=Float64),
        Field(name="cashout_count",  dtype=Int64),
        Field(name="transfer_count", dtype=Int64),
        Field(name="payment_count",  dtype=Int64),
        Field(name="cashout_ratio",  dtype=Float64),
        Field(name="transfer_ratio", dtype=Float64),
    ], source=customer_source,)

customer_risk_fv = FeatureView(
    name="customer_risk",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="balance_drain_count", dtype=Int64),
        Field(name="balance_drain_ratio", dtype=Float64),
        Field(name="avg_balance_before",  dtype=Float64),
        Field(name="past_fraud_count",    dtype=Int64),
        Field(name="fraud_rate",          dtype=Float64),
        Field(name="dest_unchanged_ratio",dtype=Float64),
        Field(name="was_flagged",         dtype=Int64),
   ], source=customer_source,)

fraud_detection_svc = FeatureService(
    name="fraud_detection_v1",
    features=[
        customer_behavior_fv[["txn_count","avg_amount","max_amount","cashout_ratio","transfer_ratio"]],
        customer_risk_fv[["balance_drain_ratio","fraud_rate","was_flagged"]],],)
'''


In [113]:
with open("feature_repo/features.py", "w") as f:
    f.write(features_py)

In [114]:
for root, _, files in os.walk("feature_repo"):
    for f in files:
        print(f"  {os.path.join(root,f)}")

  feature_repo/feature_store.yaml
  feature_repo/features.py
  feature_repo/data/registry.db
  feature_repo/data/online_store.db


##Schema Register

**feast apply** bütün Entity, FeatureView və FeatureService-ləri registry-ə qeyd edir.  
Online store-da (SQLite) lazımlı cədvəlləri yaradır.


In [100]:
!pip install feast

In [67]:
!which feast

/usr/local/bin/feast


In [115]:
import subprocess

result = subprocess.run(
    ["/usr/local/bin/feast", "apply"],
    cwd="feature_repo", capture_output=True, text=True)
print(result.stdout)
print(result.stderr[:300])

No project found in the repository. Using project name paysim_fraud_store defined in feature_store.yaml
Applying changes for project paysim_fraud_store
Updated feature view customer_risk
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created"
file_options {
  uri: "/content/data/agg.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "customer_features_source"
meta {
  created_timestamp {
    seconds: 1778587816
    nanos: 815684000
  }
  last_updated_timestamp {
    seconds: 1778587816
    nanos: 815684000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created"
file_options {
  uri: "/content/data/agg.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "customer_features_source"
meta {
  created_timestamp {
    seconds: 1778589125
    nanos: 105941000
  }
  last_updated_timestamp {
    seconds: 1778589125
    nanos: 

##Materialization — Offline dan Online Store a
**feast materialize** Parquet offline store-dan SQLite/Redis online store-a məlumat köçürür.

In [116]:
# Tarix aralıq data-nın event_timestamp aralığına uyğun olmalıdır
from datetime import datetime, timedelta
cf = pd.read_parquet("data/agg.parquet")
start_dt = cf["event_timestamp"].min().strftime("%Y-%m-%dT%H:%M:%S")
end_dt   = (cf["event_timestamp"].max() + pd.Timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%S")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [117]:

result = subprocess.run(
    ["/usr/local/bin/feast", "materialize", start_dt, end_dt],
    cwd="feature_repo", capture_output=True, text=True)

skip = {"DeprecationWarning", "UserWarning", "reserializ", "entity_key"}
for line in result.stdout.splitlines():
    if not any(w in line for w in skip):
        print(line)

print("Ugurlu" if result.returncode == 0 else f"Ugursuz {result.stderr[:300]}")

Materializing 2 feature views from 2024-01-01 01:00:00+00:00 to 2024-02-01 00:00:00+00:00 into the sqlite online store.

customer_risk:
customer_behavior:
Ugurlu


In [118]:
# Online storeu yoxlayaq
import sqlite3

online_db = "feature_repo/data/online_store.db"
conn = sqlite3.connect(online_db)
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()

#Mezmunu
for (t,) in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"  {t}: {n:,} sıra")
conn.close()
print("get_online_features() işləyir")

  paysim_fraud_store_customer_behavior: 523,917 sıra
  paysim_fraud_store_customer_risk: 407,491 sıra
get_online_features() işləyir


### Feature Store Python Client

In [119]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo/")
print(store.project)

print("FeatureViewlar")
for fv in store.list_feature_views():
    feats = [f.name for f in fv.features]
    print(f"  [{fv.name}]  TTL={fv.ttl}  features={feats}")

print("FeatureServicelər")
for fs in store.list_feature_services():
    print(f"  [{fs.name}]  {fs.description}")

paysim_fraud_store
FeatureViewlar
  [customer_risk]  TTL=30 days, 0:00:00  features=['balance_drain_count', 'balance_drain_ratio', 'avg_balance_before', 'past_fraud_count', 'fraud_rate', 'dest_unchanged_ratio', 'was_flagged']
  [customer_behavior]  TTL=30 days, 0:00:00  features=['txn_count', 'total_amount', 'avg_amount', 'max_amount', 'cashout_count', 'transfer_count', 'payment_count', 'cashout_ratio', 'transfer_ratio']
FeatureServicelər
  [fraud_detection_v1]  


## Offline Store Training Dataset

get_historical_features() — point-in-time correct feature-lar qaytarır.

Bu, data leakage-in qarşısını alır:  
Müştəri X üçün **event_timestamp = 2024-02-01** versəm,  
Feast yalnız həmin tarixdən əvvəl baş vermiş feature dəyərlərini qaytarır.

In [120]:
#Entity DataFrame kimin hansı anda featureların istəyirik
#Bütün unikallara ən son timestamp
entity_df = pd.DataFrame({
    "customer_id":     agg["customer_id"].tolist(),

    # event_timestamp datanın max timestamından bir az sonra (TTL window içində)
    "event_timestamp": pd.read_parquet("data/agg.parquet")["event_timestamp"].max() + pd.Timedelta(hours=1),})


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [121]:
entity_df.shape

(58213, 2)

In [122]:
entity_df.head(3)

,customer_id,event_timestamp
0,C1000013879,2024-02-01 00:00:00+00:00
1,C1000036340,2024-02-01 00:00:00+00:00
2,C1000048287,2024-02-01 00:00:00+00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [123]:

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        # customer_behavior view-dan
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        # customer_risk view-dan
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
        "customer_risk:fraud_rate",
        "customer_risk:was_flagged",
    ],
).to_df()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [124]:
training_df.shape

(53359, 11)

In [127]:
training_df.drop(columns=["event_timestamp"]).head(5)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,customer_id,txn_count,avg_amount,max_amount,cashout_ratio,transfer_ratio,balance_drain_ratio,dest_unchanged_ratio,fraud_rate,was_flagged
0,C173320985,1,15321.41,15321.41,0.0,0.0,0.0,1.0,0.0,0
1,C654374397,1,229633.63,229633.63,0.0,0.0,0.0,0.0,0.0,0
2,C1638128429,1,40948.58,40948.58,0.0,0.0,1.0,1.0,0.0,0
3,C1094160807,1,16903.22,16903.22,0.0,0.0,0.0,1.0,0.0,0
4,C1443756363,1,10313.00,10313.00,0.0,0.0,0.0,1.0,0.0,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [128]:
training_df.isnull().sum()

,0
customer_id,0
event_timestamp,0
txn_count,0
avg_amount,0
max_amount,0
cashout_ratio,0
transfer_ratio,0
balance_drain_ratio,0
dest_unchanged_ratio,0
fraud_rate,0


In [132]:
feat_cols = [c
for c in training_df.columns
      if c not in ["customer_id","event_timestamp"]]

training_df[feat_cols].head(3)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,txn_count,avg_amount,max_amount,cashout_ratio,transfer_ratio,balance_drain_ratio,dest_unchanged_ratio,fraud_rate,was_flagged
0,1,15321.41,15321.41,0.0,0.0,0.0,1.0,0.0,0
1,1,229633.63,229633.63,0.0,0.0,0.0,0.0,0.0,0
2,1,40948.58,40948.58,0.0,0.0,1.0,1.0,0.0,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [133]:
training_df[feat_cols].describe().round(3)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,txn_count,avg_amount,max_amount,cashout_ratio,transfer_ratio,balance_drain_ratio,dest_unchanged_ratio,fraud_rate,was_flagged
count,53359.0,5.335900e+04,5.335900e+04,53359.000,53359.000,53359.000,53359.000,53359.000,53359.000
mean,1.0,3.769069e+05,3.769069e+05,0.373,0.146,0.632,0.388,0.149,0.000
std,0.0,1.207580e+06,1.207580e+06,0.484,0.353,0.482,0.487,0.356,0.017
min,1.0,0.000000e+00,0.000000e+00,0.000,0.000,0.000,0.000,0.000,0.000
25%,1.0,1.669017e+04,1.669017e+04,0.000,0.000,0.000,0.000,0.000,0.000
50%,1.0,9.834822e+04,9.834822e+04,0.000,0.000,1.000,0.000,0.000,0.000
75%,1.0,2.591864e+05,2.591864e+05,1.000,0.000,1.000,1.000,0.000,0.000
max,1.0,6.988673e+07,6.988673e+07,1.000,1.000,1.000,1.000,1.000,1.000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## ML Model Training

Training feature-ları hazırdır. Fraud risk score modeli train edirik.

**Label:** fraud_rate > 0 olan müştəri fraud keçmişinə malikdir.

In [138]:
feature_cols = [
    "txn_count", "avg_amount", "max_amount",
    "cashout_ratio", "transfer_ratio",
    "balance_drain_ratio", "dest_unchanged_ratio",]

In [140]:
training_df[feature_cols + ["fraud_rate"]].isna().sum()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,0
txn_count,0
avg_amount,0
max_amount,0
cashout_ratio,0
transfer_ratio,0
balance_drain_ratio,0
dest_unchanged_ratio,0
fraud_rate,0


In [141]:
df_ml = training_df[feature_cols + ["fraud_rate"]].copy()
df_ml["is_fraud_customer"] = (df_ml["fraud_rate"] > 0).astype(int)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [142]:
len(df_ml)

53359

In [143]:
df_ml['is_fraud_customer'].sum()

np.int64(7948)

In [149]:
print(f"{df_ml['is_fraud_customer'].mean()*100:.1f}%")

14.9%


In [152]:
from sklearn.model_selection import train_test_split
X = df_ml[feature_cols]
y = df_ml["is_fraud_customer"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() > 1 else None)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [153]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


GradientBoostingClassifier(random_state=42)

In [156]:
# Feature importances
imps = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:")
for feat, imp in imps.items():
    bar = "o" * int(imp * 60)
    print(f"  {feat:<25}  {imp*100:5.1f}%  {bar}")



Feature Importances:
  dest_unchanged_ratio        30.3%  oooooooooooooooooo
  avg_amount                  21.0%  oooooooooooo
  cashout_ratio               18.6%  ooooooooooo
  transfer_ratio              16.4%  ooooooooo
  max_amount                  13.1%  ooooooo
  balance_drain_ratio          0.5%  
  txn_count                    0.0%  


In [157]:
# Model qiymətləndirməsi
from sklearn.metrics import roc_auc_score, classification_report
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = model.predict(X_test)

if y_test.nunique() > 1:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC: {auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Normal","Risk"]))
else:
    print(f"Risk score aralıq: {y_proba.min():.4f} — {y_proba.max():.4f}")


ROC-AUC: 0.9666

              precision    recall  f1-score   support

      Normal       0.96      1.00      0.98      9082
        Risk       0.98      0.73      0.84      1590

    accuracy                           0.96     10672
   macro avg       0.97      0.87      0.91     10672
weighted avg       0.96      0.96      0.96     10672



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##Online Store — Real-time Inference

get_online_features() — production APIda işlənir.  
SQLite/Redis-dən millisaniyə latency ilə featureları qaytarır.

**Axış:** [API: customer_id] - get_online_features() - model.predict() - risk_score

In [161]:
import time

# Test müştəriləri (real APIda request-dən gəlir)
test_customers = agg["customer_id"].tolist()[:5]
test_customers

['C1000013879', 'C1000036340', 'C1000048287', 'C100005244', 'C1000074914']

In [163]:
t0 = time.time()
online_feat = store.get_online_features(
    features=[
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
    ],
    entity_rows=[{"customer_id": cid} for cid in test_customers],
).to_df()
latency = (time.time() - t0) * 1000

In [164]:
print(f"Latency: {latency:.1f}ms  ({len(test_customers)} clients)")

Latency: 25.6ms  (5 clients)


In [165]:
online_feat

,customer_id,avg_amount,cashout_ratio,max_amount,transfer_ratio,txn_count,dest_unchanged_ratio,balance_drain_ratio
0,C1000013879,516459.01,1.0,516459.01,0.0,1,0.0,1.0
1,C1000036340,253648.68,0.0,253648.68,1.0,1,1.0,1.0
2,C1000048287,16669.78,0.0,16669.78,0.0,1,1.0,0.0
3,C100005244,99891.36,0.0,99891.36,0.0,1,0.0,0.0
4,C1000074914,315857.55,1.0,315857.55,0.0,1,0.0,1.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [166]:
# Risk score hesablayaq
X_infer = online_feat[feature_cols].fillna(online_feat[feature_cols].median())
scores  = model.predict_proba(X_infer)[:, 1]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [169]:
for cid, score in zip(test_customers, scores):
    label = "yuksek — Blok" if score > 0.6 else "orta — İzlə" if score > 0.3 else "az — Keç"
    print(f"  {cid[:15]:<16}  {score:.4f}  {label}")


  C1000013879       0.2585  az — Keç
  C1000036340       0.9931  yuksek — Blok
  C1000048287       0.0003  az — Keç
  C100005244        0.0004  az — Keç
  C1000074914       0.1048  az — Keç


In [172]:
# FeatureService ilə online sorğu
fraud_svc = store.get_feature_service("fraud_detection_v1")
online_via_svc = store.get_online_features(
    features=fraud_svc,
    entity_rows=[{"customer_id": cid} for cid in test_customers[:3]],
).to_df()

print(f"Sutunlar: {online_via_svc.columns.tolist()}")


Sutunlar: ['customer_id', 'avg_amount', 'cashout_ratio', 'max_amount', 'transfer_ratio', 'txn_count', 'was_flagged', 'fraud_rate', 'balance_drain_ratio']
